<a href="https://colab.research.google.com/github/Mizharrrrrhidi1818/EnsembleMethod-FusionMethod/blob/main/Notebooks/Ensemble_Classifier_Fusion_on_Amazon_Sales_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Ensemble Classifier Fusion on Amazon Sales Data**

# **1. Introduction**

Predictive modeling in e-commerce rarely relies on a single algorithm. Different models capture different patterns: tree-based methods handle non-linear feature interactions well, distance-based methods excel in dense feature spaces, and linear models provide stable decision boundaries. When these models disagree, especially on ambiguous or borderline samples, raw predictions become unreliable.

Classifier fusion addresses this by combining multiple model outputs into a single, more robust decision. This project implements a complete fusion pipeline across three distinct prediction levels (abstract, rank, and measurement) using a real-world Amazon sales dataset. The goal is to demonstrate how different combination rules perform when faced with a deliberately selected ambiguous test case, providing practical insight into ensemble design for production systems.

# **2. Methodology**

The workflow follows a structured, multi-stage pipeline:

1. Data Preparation: Raw Amazon sales records are cleaned, encoded, scaled, and split into training and testing sets.
2. Base Model Training: Six diverse classifiers are trained independently. Training error rates are recorded to enable performance-weighted fusion.
3. Ambiguity Selection: Instead of picking a random test sample, the instance with the highest prediction entropy across the ensemble is isolated. This guarantees maximum disagreement among base models, creating a rigorous test case for fusion rules.
4. Prediction Extraction: For the selected sample, three output formats are extracted:
   - Abstract: Hard class labels from `predict()`
   - Rank: Ordinal positions derived from probability ordering
   - Measurement: Full probability vectors from `predict_proba()`
5. Fusion Execution: Each level is processed through multiple combination rules. Abstract methods aggregate hard votes, rank methods aggregate positional information, and measurement methods aggregate continuous confidence scores.
6. Evaluation: All fused decisions are compared against the ground-truth label to assess which combination strategy handles ambiguity most effectively.

The pipeline is fully deterministic (fixed random states, stratified splits) and adapts dynamically to the number of classes present after cleaning.

# **3. Objective**

- Evaluate how different fusion levels (abstract, rank, measurement) handle classifier disagreement on a single ambiguous e-commerce sample.
- Compare traditional voting schemes against information-theoretic and evidence-based combination rules.
- Identify which fusion strategy consistently yields correct decisions when base models are uncertain.
- Provide a reproducible, production-ready reference for implementing classifier fusion in tabular sales data contexts.

# **4. Data Exploration and Preprocessing**

## **4.1. Dataset Overview**

The Amazon sales dataset contains product-level information typical of e-commerce platforms. Columns generally include pricing (`actual_price`, `discounted_price`), discount metrics, rating counts, stock status, bestseller flags, and product categories. The target variable is automatically detected (preferring `category` or `purchased`), and the dataset is loaded directly via Kaggle's API. Initial inspection shows a mix of numeric, categorical, and string-formatted features, with occasional missing values and non-standard character encodings (e.g., currency symbols, commas).

In [ ]:
# Load dataset anda data preprocessing
import kagglehub

# Download latest version
path = kagglehub.dataset_download("karkavelrajaj/amazon-sales-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'amazon-sales-dataset' dataset.
Path to dataset files: /kaggle/input/amazon-sales-dataset


In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# This know our dataset after we have finished download

/kaggle/input/amazon-sales-dataset/amazon.csv


In [ ]:
import os
import numpy as np
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# Use the 'path' variable from the previous download cell to construct the correct file path
df = pd.read_csv(os.path.join(path, "amazon.csv"))

df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [ ]:
print(f"Dataset loaded. Shape: {df.shape}")

Dataset loaded. Shape: (1465, 16)


## **4.2. Data Cleaning and Feature Engineering**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

print("="*60)
print("STARTING DATA PREPROCESSING PIPELINE")
print("="*60)
print(f"Initial Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")


# 1. Column Cleanup & Target Selection
drop_cols = [c for c in df.columns if 'id' in c.lower() or 'url' in c.lower()]
print(f"[1]  Dropping ID/URL columns: {drop_cols if drop_cols else 'None found'}")
df_clean = df.drop(columns=drop_cols, errors='ignore')

# Identify target: prefer 'category', 'purchased', or first categorical column
target_candidates = ['category', 'purchased', 'best_seller', 'stock_status']
target_col = next((c for c in target_candidates if c in df_clean.columns),
                  df_clean.select_dtypes(include=['object', 'category']).columns[0] if not df_clean.select_dtypes(include=['object', 'category']).empty else None)

# Handle case where no categorical columns are found for target
if target_col is None:
    raise ValueError("No suitable target column found. Please check data or specify target_col manually.")
print(f"Selected Target Column: '{target_col}'\n")

# Separate features(X) & target(y)
y = df_clean[target_col].copy()
X = df_clean.drop(columns=[target_col])
print(f"Split: Features (X) = {X.shape} | Target (y) = {y.shape}")

# 2: String-to-Numeric Conversion
def clean_and_convert_price(series):
    return series.astype(str).str.replace('₹', '').str.replace(',', '').astype(float)

def clean_and_convert_percentage(series):
    return series.astype(str).str.replace('%', '').astype(float)

def clean_and_convert_rating_count(series):
    return series.astype(str).str.replace(',', '').astype(float)

cleaned_cols = []
if 'discounted_price' in X.columns:
    X['discounted_price'] = clean_and_convert_price(X['discounted_price'])
    cleaned_cols.append('discounted_price')
if 'actual_price' in X.columns:
    X['actual_price'] = clean_and_convert_price(X['actual_price'])
    cleaned_cols.append('actual_price')
if 'discount_percentage' in X.columns:
    X['discount_percentage'] = clean_and_convert_percentage(X['discount_percentage'])
    cleaned_cols.append('discount_percentage')
if 'rating_count' in X.columns:
    X['rating_count'] = clean_and_convert_rating_count(X['rating_count'])
    cleaned_cols.append('rating_count')

if cleaned_cols:
    print(f"\n[2] Cleaned & converted to float: {cleaned_cols}")
else:
    print(f"\n[2] No price/percentage/rating columns found to clean.")

# 3. Missing Value Imputation
print(f"\n[3] Missing Values BEFORE Imputation:")
missing_before = X.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
if missing_before.empty:
    print("   None found.")
else:
    print(missing_before.to_string())

# Impute missing values
num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

# Only impute if there are numerical columns to impute
if not num_cols.empty:
    X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
    print(f"Imputed {len(num_cols)} numeric column(s) with median strategy.")
else:
    print(f"No numeric columns found for imputation.")

# Only impute if there are categorical columns to impute
if not cat_cols.empty:
    X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])
    print(f"Imputed {len(cat_cols)} categorical column(s) with mode strategy.")
else:
    print(f"No categorical columns found for imputation.")

# 4: Target & Feature Encoding
# Encode categoricals
le = LabelEncoder()
y = le.fit_transform(y)
CLASSES = le.classes_
C = len(CLASSES)
print(f"\n[4]Target Encoded: {C} unique classes → {list(CLASSES)}")

# 5: Rare Class Filtering & Re-Encoding
# Only apply one-hot encoding if there are categorical columns after imputation
print(f"\n[5] Feature Encoding (One-Hot)...")
if not cat_cols.empty:
    pre_dummies_cols = X.shape[1]
    X = pd.get_dummies(X, columns=cat_cols, drop_first=False).astype(float)
    print(f"   Encoded {len(cat_cols)} categorical column(s).")
    print(f"   Dimensions expanded: {pre_dummies_cols} → {X.shape[1]} features.")
else:
    X = X.astype(float)
    print(f"   No categorical features to encode. All features already numeric.")

# Identify and remove classes with fewer than 6 members (to ensure >=3 in training set after split)
print(f"\n[6] Checking Class Distribution (Pre-Filter):")
class_counts = pd.Series(y).value_counts().sort_index()
print(class_counts.to_string())

classes_to_remove = class_counts[class_counts < 6].index

if not classes_to_remove.empty:
    # Map encoded indices back to original class names for readability
    removed_names = [CLASSES[idx] for idx in classes_to_remove]
    print(f"\n   Removing rare classes (<6 samples): {removed_names}")

    # Get boolean mask for rows to keep
    rows_to_keep_mask = ~pd.Series(y).isin(classes_to_remove)
    X = X[rows_to_keep_mask]
    y = y[rows_to_keep_mask.values] # Use .values to match numpy array if y is numpy array

    # Re-encode y and update CLASSES and C if classes were removed
    le_filtered = LabelEncoder()
    y = le_filtered.fit_transform(y)
    CLASSES = le_filtered.classes_
    C = len(CLASSES)
    print(f"   Dataset filtered. New shape: X={X.shape}, y={y.shape}")
    print(f"   Updated Classes: {list(CLASSES)}")
else:
    print(f"   All classes have ≥6 samples. No filtering required.")

print(f"\n" + "="*60)
print(" PREPROCESSING COMPLETE!")
print(f"  • Final Classes (C): {C}")
print(f"  • Class Labels: {list(CLASSES)}")
print(f"  • Features Shape (X): {X.shape}")
print(f"  • Target Shape (y): {y.shape}")
print("="*60)

STARTING DATA PREPROCESSING PIPELINE
Initial Dataset Shape: 1465 rows × 16 columns

[1]  Dropping ID/URL columns: ['product_id', 'user_id', 'review_id']
Selected Target Column: 'category'

Split: Features (X) = (1465, 12) | Target (y) = (1465,)

[2] Cleaned & converted to float: ['discounted_price', 'actual_price', 'discount_percentage', 'rating_count']

[3] Missing Values BEFORE Imputation:
rating_count    2
Imputed 4 numeric column(s) with median strategy.
Imputed 8 categorical column(s) with mode strategy.

[4]Target Encoded: 211 unique classes → ['Car&Motorbike|CarAccessories|InteriorAccessories|AirPurifiers&Ionizers', 'Computers&Accessories|Accessories&Peripherals|Adapters|USBtoUSBAdapters', 'Computers&Accessories|Accessories&Peripherals|Audio&VideoAccessories|PCHeadsets', 'Computers&Accessories|Accessories&Peripherals|Audio&VideoAccessories|PCMicrophones', 'Computers&Accessories|Accessories&Peripherals|Audio&VideoAccessories|PCSpeakers', 'Computers&Accessories|Accessories&Periphe

## **4.3. Train Model**

In [ ]:
# 1. Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 2. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Model Initialization
print("\nTraining classifiers...")
raw_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
}

# 4. Training Loop & Error Calculation
MODELS = {}
ERROR = {}
for name, clf in raw_models.items():
    clf.fit(X_train_scaled, y_train)
    MODELS[name] = clf
    acc = accuracy_score(y_train, clf.predict(X_train_scaled))
    ERROR[name] = 1.0 - acc  # Training error rate for weighting
    print(f"  {name:<25} | Train Acc: {acc:.4f}")


Training classifiers...
  Decision Tree             | Train Acc: 0.4342
  Naive Bayes               | Train Acc: 1.0000
  KNN                       | Train Acc: 0.1290
  Logistic Regression       | Train Acc: 1.0000


## **4.4. Select One Ambiguous Test Object**

To stress-test the fusion pipeline, the most uncertain test sample is selected programmatically. For each test instance, the ensemble's probability outputs are averaged. Shannon entropy is calculated across these averaged probabilities

The index with maximum entropy is chosen. High entropy indicates flat or conflicting probability distributions across the ensemble, meaning no single model is confident. The corresponding test feature vector (x_sample) and true label (TRUE_LABEL) are stored. The true label is never used during fusion calculation; it is reserved solely for post-hoc evaluation.

In [ ]:
# Select One Ambiguous Test Object
# Ambiguity = highest entropy of averaged predictions
# Ensemble Probability Averaging
avg_proba = np.mean([clf.predict_proba(X_test_scaled) for clf in MODELS.values()], axis=0)

# Shannon Entropy Calculation
entropy = -np.sum(avg_proba * np.log(avg_proba + 1e-12), axis=1)

# Select the Most Ambiguous Sample
ambiguous_idx = np.argmax(entropy)

# Extract Sample with Preserved Shape
x_sample = X_test_scaled[ambiguous_idx:ambiguous_idx+1]  # Shape (1, n_features)

# Retrieve Ground Truth Label
TRUE_LABEL = CLASSES[y_test[ambiguous_idx]]

# Diagnostic Output
print(f"\nSelected ambiguous test sample index: {ambiguous_idx}")
print(f"True class: {TRUE_LABEL}")
print(f"Features shape: {x_sample.shape}, Classes: {C}")


Selected ambiguous test sample index: 331
True class: 182
Features shape: (1, 9139), Classes: 56


# **5. Collection Three Prediction Level**

Before fusion, each model's output for the ambiguous sample is transformed into three distinct representations:
- Abstract Level: Each classifier's `predict()` call returns a single hard label. This is the most interpretable but least informative format, discarding confidence margins.
- Measurement Level: `predict_proba()` returns a probability vector of length C (number of classes). Each value represents the model's confidence that the sample belongs to that class. These vectors form the Decision Profile matrix.
- Rank Level: Probability vectors are sorted in descending order. Each class receives a rank from 1 (highest probability) to `C` (lowest). This ordinal representation is scale-invariant and useful when absolute confidence calibration varies between models.

All three representations are stored in dictionaries keyed by model name, enabling parallel processing across fusion stages. The rank and measurement levels retain full class ordering, while the abstract level collapses it to a single vote.

In [ ]:
def sub(title):
    print(f"\n{'=' * 5} {title.upper()} {'=' * 5}")

In [ ]:
import numpy as np

# Measurement: probability vector per classifier (2 Dimensions)
PROBA = {name: clf.predict_proba(x_sample)[0]
         for name, clf in MODELS.items()}

# Abstract: hard label per classifier (1 Dimension)
ABSTRACT = {name: CLASSES[clf.predict(x_sample)[0]]
            for name, clf in MODELS.items()}

# Rank level Construction
RANK = {}
for name, p in PROBA.items():
    order = np.argsort(-p)                  # indices best→worst
    r = np.empty(C, dtype=int)
    for pos, ci in enumerate(order):
        r[ci] = pos + 1
    RANK[name] = r                          # r[class_index] = rank

# Structured Output Formatting
sub("Abstract level — one hard label per classifier")
for name, label in ABSTRACT.items():
    print(f"  {name:<25} → {{{label}}}")

sub("Measurement level — probability vector per classifier")
for name, p in PROBA.items():
    vals = tuple(round(float(v), 4) for v in p)
    print(f"  {name:<25} → {vals}")

sub("Rank level — rank per class (1 = most likely)")
header_str = f"  {'Model':<25} " + " ".join(f"{c:<12}" for c in CLASSES)
print(header_str)
for name, r in RANK.items():
    row_vals = " ".join(f"{r[i]:<12}" for i in range(C))
    print(f"  {name:<25} {row_vals}")


===== ABSTRACT LEVEL — ONE HARD LABEL PER CLASSIFIER =====
  Decision Tree             → {140}
  Naive Bayes               → {82}
  KNN                       → {117}
  Logistic Regression       → {10}

===== MEASUREMENT LEVEL — PROBABILITY VECTOR PER CLASSIFIER =====
  Decision Tree             → (0.0, 0.0081, 0.0, 0.0242, 0.0081, 0.0, 0.0242, 0.0081, 0.0081, 0.0081, 0.0242, 0.0242, 0.0081, 0.0, 0.0565, 0.0081, 0.0242, 0.0081, 0.0323, 0.0081, 0.0, 0.0, 0.0403, 0.0, 0.0242, 0.0, 0.0, 0.0081, 0.0, 0.0403, 0.0565, 0.0645, 0.0323, 0.0, 0.0887, 0.0, 0.0, 0.0081, 0.0242, 0.0242, 0.0565, 0.0323, 0.0161, 0.0081, 0.0161, 0.0645, 0.0323, 0.0, 0.0, 0.0403, 0.0, 0.0403, 0.0, 0.0, 0.0, 0.0)
  Naive Bayes               → (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6544, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3456, 0.0, 0.0, 

1. Abstracl Level <br>
Abstract Level have format single class label per modal, its purpose was hard voting input:
   - All four models predict different classes for the same sample.
   - There is zero consensus at the hard-decision level.
   - A simple majority vote would fail (1 vote each → tie). <br>

   This is the definition of ambiguity: no single model is confidently correct, and their reasoning paths diverge completely.

2. Measurement Level: Confidence Patterns Revealed <br>
  a. Decision Tree Probabilities (Sparse, Fragmented):
   - Many exact zeros (0.0): tree leaf contains no training samples of those classes
   - Small non-zero values (0.0081, 0.0242): classes present in sibling leaves (smoothing)
   - Highest value 0.0887 at index 34

   b. Naive Bayes Probabilities (Extremely Concentrated):
   - Only two non-zero probabilities: 0.6544 and 0.3456
   - All other 54 classes assigned exactly 0.0
   - Sum = 1.0

   c. KNN Probabilities (Bimodal Local Uncertainty):
   - Only two non-zero values: 0.4 for class 10, 0.6 for class 117
   - Reflects the 5 nearest neighbors: ~3 voted class 117, ~2 voted class 10
   - Honest representation of local ambiguity. The sample sits near a decision boundary between two categories.

   d. Logistic Regression Probabilities (Smooth, Distributed)
   - All 56 classes receive non-zero probability
   - Highest: 0.4552 for class 10 (matches abstract prediction)
   - Others range 0.0075–0.0275 → low but non-negligible confidence

3. Rank Level: Ordinal Preferences <br>
The header lists 56 class labels (e.g., 10, 13, 15, ... 201). Each row shows the rank (1 = most likely) each model assigned to that class.







# **6. Abstract Level Fusion**

Abstract level fusion is the simplest way to combine multiple models. It completely ignores how confident each model is. It only looks at their final, hard guess (the class label). We can imagine that think of it like a panel of judges holding up scorecards that only say "Class A" or "Class B". The fusion system acts as a referee, counting the cards and picking the most frequent one.

Our pipeline tests two abstract methods: Majority Vote (equal trust) and Weighted Majority Vote (trust based on past performance).


In [ ]:
# 6a. Majority Vote
sub("2a. Majority Vote")
print("  Formula: d̂(x) = argmax_v  Σᵢ I(dᵢ(x) = v)\n")
vote_counts = {cls: 0 for cls in CLASSES}
for label in ABSTRACT.values():
    vote_counts[label] += 1

print("  Votes cast by each classifier:")
for name, label in ABSTRACT.items():
    print(f"    {name:<25} → votes for '{label}'")
print(f"\n  Vote tally:")
for cls, cnt in vote_counts.items():
    print(f"    {cls:<12} : {cnt} vote(s)")

mv_winner = max(vote_counts, key=vote_counts.get)
print(f"\n  Decision (Majority Vote)  →  {{{mv_winner}}}")

# 6b. Weighted Majority Vote
sub("2b. Weighted Majority Vote")
print("  Formula: d̂(x) = argmax_v  Σᵢ wᵢ·I(dᵢ(x) = v)")
print("  Weights: wᵢ ∝ log((1 − eᵢ) / eᵢ)\n")

raw_w = {}
for name, e in ERROR.items():
    e = min(max(e, 1e-4), 1 - 1e-4)
    raw_w[name] = np.log((1 - e) / e)

total_w = sum(raw_w.values())
W = {name: raw_w[name] / total_w for name in MODELS}

print("  Weights:")
for name, w in W.items():
    print(f"    {name:<25} err={ERROR[name]:.4f}  w={w:.4f}")

wmv_scores = {cls: 0.0 for cls in CLASSES}
print("\n  Weighted votes:")
for name, label in ABSTRACT.items():
    wmv_scores[label] += W[name]
    print(f"    {name:<25} votes '{label}' with weight {W[name]:.4f}")

print(f"\n  Weighted score per class:")
for cls, sc in wmv_scores.items():
    print(f"    {cls:<12} : {sc:.4f}")

wmv_winner = max(wmv_scores, key=wmv_scores.get)
print(f"\n  Decision (Weighted Majority Vote)  →  {{{wmv_winner}}}")



===== 2A. MAJORITY VOTE =====
  Formula: d̂(x) = argmax_v  Σᵢ I(dᵢ(x) = v)

  Votes cast by each classifier:
    Decision Tree             → votes for '140'
    Naive Bayes               → votes for '82'
    KNN                       → votes for '117'
    Logistic Regression       → votes for '10'

  Vote tally:
    10           : 1 vote(s)
    13           : 0 vote(s)
    15           : 0 vote(s)
    16           : 0 vote(s)
    18           : 0 vote(s)
    20           : 0 vote(s)
    23           : 0 vote(s)
    28           : 0 vote(s)
    39           : 0 vote(s)
    42           : 0 vote(s)
    49           : 0 vote(s)
    51           : 0 vote(s)
    58           : 0 vote(s)
    71           : 0 vote(s)
    76           : 0 vote(s)
    77           : 0 vote(s)
    82           : 1 vote(s)
    89           : 0 vote(s)
    93           : 0 vote(s)
    94           : 0 vote(s)
    97           : 0 vote(s)
    98           : 0 vote(s)
    103          : 0 vote(s)
    104          :

1. Majority Vote: The Simple Count
   - Each model casts exactly 1 vote for its predicted class.
   - The code tallies votes across all 56 possible classes.
   - Result: 10: 1, 82: 1, 117: 1, 140: 1. All others: 0.
   - `{10}`: This is arbitrary. If the class order changed, a different class would win. Majority vote fails when models strongly disagree.

2. Weighted Majority Vote: The Trust-Adjusted Count
   - Calculating Weights:
     - The formula log((1-e)/e) produces a positive number when error < 0.5 (model is better than random guessing).
     - It produces a negative number when error > 0.5 (model is worse than random guessing).
     - Decision Tree and KNN had training errors above 50%, meaning they were actively confusing classes. The math assigns them negative weights, which effectively means: "Subtract points from whatever class these models pick."
    
    -  Casting Weighted Votes:<br>
      Each model's vote is multiplied by its weight:
       - Class 10 (LR) → +0.5669
       - Class 82 (NB) → +0.5669
       - Class 117 (KNN) → -0.1175
       - Class 140 (DT) → -0.016
    
    - Picking the winner<br>
     The code sums scores per class. Only the four predicted classes have non-zero scores:
       - 10 → 0.5669
       - 82 → 0.5669
       - 117 → -0.1175
       - 140 → -0.0163

       Classes 10 and 82 are exactly equal. Again, max() picks the first one in the sorted list → {10}.

# **7. Rank Level Fusion**

In [ ]:
# 7a. Borda Count
sub("7a. Borda Count")
print("  Formula: Borda(vᵢ) = Σⱼ (c − rankⱼ(vᵢ))\n")
borda = np.zeros(C)
print("  Borda votes per classifier:")
for name, r in RANK.items():
    votes = C - r
    borda += votes
    vals  = {CLASSES[i]: int(votes[i]) for i in range(C)}
    print(f"    {name:<25} → {vals}")

borda_dict = {CLASSES[i]: int(borda[i]) for i in range(C)}
print(f"\n  Total Borda scores  →  {borda_dict}")
bc_winner  = CLASSES[int(np.argmax(borda))]
print(f"  Decision (Borda Count)  →  {{{bc_winner}}}")

# 7b. Highest Rank
sub("7b. Highest Rank Method")
print("  Formula: minRank(vᵢ) = min over all classifiers of rankⱼ(vᵢ)\n")
rank_mat = np.stack(list(RANK.values()), axis=0)
min_ranks = rank_mat.min(axis=0)
min_rank_dict = {CLASSES[i]: int(min_ranks[i]) for i in range(C)}
print(f"  Min rank per class  →  {min_rank_dict}")
hr_winner = CLASSES[int(np.argmin(min_ranks))]
print(f"  Decision (Highest Rank)  →  {{{hr_winner}}}")

# 7c. Intersection Method
sub("7c. Intersection Method")
print("  Training: threshold = worst rank ever given to correct class\n")
thresholds = {}
for name, clf in MODELS.items():
    proba_train = clf.predict_proba(X_train_scaled)
    worst = 0
    for i, true_ci in enumerate(y_train):
        p = proba_train[i]
        order  = np.argsort(-p)
        rank_true = int(np.where(order == true_ci)[0][0]) + 1
        if rank_true > worst: worst = rank_true
    thresholds[name] = worst

print("  Thresholds (worst training rank per classifier):")
for name, thr in thresholds.items():
    print(f"    {name:<25} threshold = {thr}")

accepted_sets = {}
print("\n  Accepted class sets for test sample:")
for name, r in RANK.items():
    thr = thresholds[name]
    accepted = {CLASSES[i] for i in range(C) if r[i] <= thr}
    accepted_sets[name] = accepted
    print(f"    {name:<25} (thr={thr}) → {accepted}")

intersection = set(CLASSES)
for s in accepted_sets.values():
    intersection &= s
if not intersection: intersection = set(CLASSES)

print(f"\n  Intersection  →  {intersection}")
inter_winner = ', '.join(sorted([str(c) for c in intersection])) # Convert to string before joining
print(f"  Decision (Intersection)  →  {{{inter_winner}}}")

# 7d. Union Method
sub("7d. Union Method")
print("  Training: max-min threshold procedure\n")
union_thresholds = {}
for name, clf in MODELS.items():
    proba_train = clf.predict_proba(X_train_scaled)
    correct_ranks = []
    for i, true_ci in enumerate(y_train):
        order     = np.argsort(-proba_train[i])
        rank_true = int(np.where(order == true_ci)[0][0]) + 1
        correct_ranks.append(rank_true)
    min_rank = min(correct_ranks)
    filtered = [r if r == min_rank else 0 for r in correct_ranks]
    union_thresholds[name] = max(filtered) if any(f > 0 for f in filtered) else 1

print("  Thresholds (max-min procedure):")
for name, thr in union_thresholds.items():
    print(f"    {name:<25} threshold = {thr}")

union_sets = {}
print("\n  Accepted class sets for test sample:")
for name, r in RANK.items():
    thr = union_thresholds[name]
    accepted = {CLASSES[i] for i in range(C) if r[i] <= thr}
    union_sets[name] = accepted
    print(f"    {name:<25} (thr={thr}) → {accepted}")

union_result = set()
for s in union_sets.values():
    union_result |= s

print(f"\n  Union  →  {union_result}")
print(f"  Decision (Union)  →  {{{', '.join(sorted([str(c) for c in union_result]))}}}") # Convert to string before joining


===== 7A. BORDA COUNT =====
  Formula: Borda(vᵢ) = Σⱼ (c − rankⱼ(vᵢ))

  Borda votes per classifier:
    Decision Tree             → {np.int64(10): 17, np.int64(13): 24, np.int64(15): 14, np.int64(16): 40, np.int64(18): 25, np.int64(20): 15, np.int64(23): 36, np.int64(28): 26, np.int64(39): 27, np.int64(42): 20, np.int64(49): 39, np.int64(51): 41, np.int64(58): 29, np.int64(71): 16, np.int64(76): 52, np.int64(77): 21, np.int64(82): 38, np.int64(89): 28, np.int64(93): 42, np.int64(94): 30, np.int64(97): 10, np.int64(98): 11, np.int64(103): 48, np.int64(104): 12, np.int64(107): 37, np.int64(114): 13, np.int64(115): 18, np.int64(116): 31, np.int64(117): 19, np.int64(119): 49, np.int64(128): 51, np.int64(135): 54, np.int64(136): 43, np.int64(139): 6, np.int64(140): 55, np.int64(141): 8, np.int64(144): 9, np.int64(156): 22, np.int64(158): 34, np.int64(159): 35, np.int64(161): 50, np.int64(162): 44, np.int64(165): 33, np.int64(166): 23, np.int64(168): 32, np.int64(169): 53, np.int64(174): 4

# **8. Measurement Level Fusion**

In [ ]:
L = len(MODELS)
DP = np.stack(list(PROBA.values()), axis=0)
names_list = list(MODELS.keys())

print(f"\n  Decision Profile DP(x)  \u2014  shape ({L} classifiers \u00d7 {C} classes)")
for i, name in enumerate(names_list):
    row = [f"{DP[i,j]:.4f}" for j in range(C)]
    print(f"  {name:<25}  {' | '.join(row)}")

def show_support(W_j, rule_name):
    w_dict = {CLASSES[j]: round(float(W_j[j]), 6) for j in range(C)}
    best   = CLASSES[int(np.argmax(W_j))]
    print(f"\n  Support W_j  \u2192  {w_dict}")
    print(f"  Decision ({rule_name})  \u2192  {{{best}}}")
    return best

# 8a. Max Rule
sub("8a. Max Rule")
print("  Formula: W_j = max over classifiers of p_{i,j}\n")
for j, cls in enumerate(CLASSES):
    vals = tuple(round(float(DP[i, j]), 4) for i in range(L))
    print(f"  col '{cls}': {vals}  \u2192  max = {max(vals):.4f}")
show_support(DP.max(axis=0), "Max Rule")

# 8b. Min Rule
sub("8b. Min Rule")
print("  Formula: W_j = min over classifiers of p_{i,j}\n")
for j, cls in enumerate(CLASSES):
    vals = tuple(round(float(DP[i, j]), 4) for i in range(L))
    print(f"  col '{cls}': {vals}  \u2192  min = {min(vals):.4f}")
show_support(DP.min(axis=0), "Min Rule")

# 8c. Median Rule
sub("8c. Median Rule")
print("  Formula: W_j = median over classifiers of p_{i,j}\n")
show_support(np.median(DP, axis=0), "Median Rule")

# 8d. Sum Rule
sub("8d. Sum Rule")
print("  Formula: W_j = \u03a3\u208a p_{i,j}\n")
show_support(DP.sum(axis=0), "Sum Rule")

# 8e. Product Rule
sub("8e. Product Rule")
print("  Formula: W_j = \u03a0\u208a p_{i,j}  (zeros \u2192 \u03b5=0.001)\n")
EPS = 1e-3
DP_safe = np.where(DP == 0, EPS, DP)
show_support(DP_safe.prod(axis=0), "Product Rule")

# 8f. Weighted Average
sub("8f. Weighted Average")
print("  Formula: W_j = \u03a3\u208a w\u208a\u00b7p_{i,j}\n")

# Clamp error rates to avoid division by zero or infinity
clamped_error = {}
for name, e in ERROR.items():
    clamped_error[name] = min(max(e, 1e-4), 1 - 1e-4) # Ensure error is not 0 or 1

raw_wa = {name: (1 - clamped_error[name]) / clamped_error[name] for name in MODELS}
total_wa = sum(raw_wa.values())
WA = np.array([raw_wa[n] / total_wa for n in names_list])
show_support(WA @ DP, "Weighted Average")

# 8g. Probabilistic Product
sub("8g. Probabilistic Product")
print("  Formula: W_j = \u03a0\u208a p_{i,j} / P(v\u207f)^(L-1)\n")
prior = np.array([(y_train == j).sum() / len(y_train) for j in range(C)])
prior = np.where(prior == 0, EPS, prior)
print(f"  Class priors P(v\u207f): {[f'{prior[j]:.3f}' for j in range(C)]}")
show_support(DP_safe.prod(axis=0) / (prior ** (L - 1)), "Probabilistic Product")

# 8h. Decision Templates
sub("8h. Decision Templates")
print("  Training: DT_j = mean of DP(x) over all training samples of class j\n")
DT = {}
for j in range(C):
    mask = (y_train == j)
    if mask.sum() == 0: continue
    dps = np.array([clf.predict_proba(X_train_scaled[mask]) for clf in MODELS.values()])
    DT[j] = dps.mean(axis=1)  # shape (L, C)

print("\n  Euclidean similarity: s = 1 \u2212 (1/L\u00b7C) \u03a3(DP \u2212 DT_j)\u00b2")
W_dt = np.zeros(C)
for j in range(C):
    if j not in DT:
        W_dt[j] = 0.0
        continue
    diff = DP - DT[j]
    sq = (diff ** 2).sum()
    W_dt[j] = 1 - sq / (L * C)
show_support(W_dt, "Decision Templates")

# 8i. Dempster-Shafer
sub("8i. Dempster-Shafer (Theory of Evidence)")
print("  Step 1: \u03c6_{j,m} = proximity of classifier m to template j\n")
phi = np.zeros((C, L))
for m in range(L):
    dists = np.array([np.linalg.norm(DT[j][m] - DP[m]) ** 2 for j in range(C)])
    inv   = 1.0 / (1.0 + dists)
    phi[:, m] = inv / inv.sum()

print("  Step 2: Bel_j(m) = belief degree\n")
bel = np.zeros((C, L))
EPS_DS = 1e-9
for j in range(C):
    for m in range(L):
        phi_jm      = phi[j, m]
        prod_others = np.prod([1 - phi[k, m] for k in range(C) if k != j])
        numerator   = phi_jm * prod_others
        denominator = 1 - phi_jm * (1 - prod_others) + EPS_DS
        bel[j, m]   = numerator / denominator

print("  Step 3: \u00b5_j = K \u00b7 \u03a0\u2081 Bel_j(m)\n")
ds_raw = np.prod(bel, axis=1)
K      = 1.0 / ds_raw.sum() if ds_raw.sum() > 0 else 1.0
show_support(K * ds_raw, "Dempster-Shafer")


  Decision Profile DP(x)  —  shape (4 classifiers × 56 classes)
  Decision Tree              0.0000 | 0.0081 | 0.0000 | 0.0242 | 0.0081 | 0.0000 | 0.0242 | 0.0081 | 0.0081 | 0.0081 | 0.0242 | 0.0242 | 0.0081 | 0.0000 | 0.0565 | 0.0081 | 0.0242 | 0.0081 | 0.0323 | 0.0081 | 0.0000 | 0.0000 | 0.0403 | 0.0000 | 0.0242 | 0.0000 | 0.0000 | 0.0081 | 0.0000 | 0.0403 | 0.0565 | 0.0645 | 0.0323 | 0.0000 | 0.0887 | 0.0000 | 0.0000 | 0.0081 | 0.0242 | 0.0242 | 0.0565 | 0.0323 | 0.0161 | 0.0081 | 0.0161 | 0.0645 | 0.0323 | 0.0000 | 0.0000 | 0.0403 | 0.0000 | 0.0403 | 0.0000 | 0.0000 | 0.0000 | 0.0000
  Naive Bayes                0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.6544 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000

np.int64(82)

# **9. Final Summary**

In [ ]:
SEP = '=' * 50
print(f"\n  True class : {TRUE_LABEL}")
print(f"  {'Method':<35} {'Decision'}")
print(f"  {'-'*50}")

results = [
    ("Abstract — Majority Vote",        mv_winner),
    ("Abstract — Weighted Maj. Vote",   wmv_winner),
    ("Rank    — Borda Count",           bc_winner),
    ("Rank    — Highest Rank",          hr_winner),
    ("Rank    — Intersection",          ', '.join(sorted([str(c) for c in intersection]))),
    ("Rank    — Union",                 ', '.join(sorted([str(c) for c in union_result]))),
    ("Measure — Max Rule",              CLASSES[int(np.argmax(DP.max(axis=0)))]),
    ("Measure — Min Rule",              CLASSES[int(np.argmax(DP.min(axis=0)))]),
    ("Measure — Median Rule",           CLASSES[int(np.argmax(np.median(DP, axis=0)))]),
    ("Measure — Sum Rule",              CLASSES[int(np.argmax(DP.sum(axis=0)))]),
    ("Measure — Product Rule",          CLASSES[int(np.argmax(DP_safe.prod(axis=0)))]),
    ("Measure — Weighted Average",      CLASSES[int(np.argmax(WA @ DP))]),
    ("Measure — Probabilistic Product", CLASSES[int(np.argmax(DP_safe.prod(axis=0) / (prior ** (L - 1))))]),
    ("Measure — Decision Templates",    CLASSES[int(np.argmax(W_dt))]),
    ("Measure — Dempster-Shafer",       CLASSES[int(np.argmax(K * ds_raw))]),
]

for method, decision in results:
    correct = "✓" if TRUE_LABEL == decision else "✗"
    print(f"  {method:<35} {{{decision}}}  {correct}")

print(f"\n{SEP}")


  True class : 182
  Method                              Decision
  --------------------------------------------------
  Abstract — Majority Vote            {10}  ✗
  Abstract — Weighted Maj. Vote       {10}  ✗
  Rank    — Borda Count               {76}  ✗
  Rank    — Highest Rank              {10}  ✗
  Rank    — Intersection              {10, 103, 104, 107, 114, 115, 116, 117, 119, 128, 13, 135, 136, 139, 140, 141, 144, 15, 156, 158, 159, 16, 161, 162, 165, 166, 168, 169, 174, 178, 18, 181, 182, 183, 187, 189, 191, 192, 20, 201, 23, 28, 39, 42, 49, 51, 58, 71, 76, 77, 82, 89, 93, 94, 97, 98}  ✗
  Rank    — Union                     {10, 117, 140, 82}  ✗
  Measure — Max Rule                  {82}  ✗
  Measure — Min Rule                  {10}  ✗
  Measure — Median Rule               {10}  ✗
  Measure — Sum Rule                  {10}  ✗
  Measure — Product Rule              {10}  ✗
  Measure — Weighted Average          {82}  ✗
  Measure — Probabilistic Product     {82}  ✗
  Measure — De

The reason there are no checkmarks is that classifier fusion is like a referee in a panel of judges. If none of the judges vote for the winner, the referee cannot magically declare them the winner. Fusion can combine votes, but it cannot invent new answers.

Here is the exact reason why Class 182 lost every time, based on the data you provided earlier.
1. The Base Models Ignored Class 182
   - Class 82 (The "fake" winner) had 65% confidence from Naive Bayes.
   - Class 10 (The other "fake" winner) had 45% confidence from Logistic Regression.
   - Class 182 (The real winner) had almost 0% support.
2. The Math Followed the Data<br>
Because the numbers for Class 182 were so low (near zero), every math rule you used buried it:
   - Sum Rule: Added up the tiny numbers (0.04+0+0+0.008). Total was still tiny compared to Class 10 or 82.
   - Product Rule: Multiplied the numbers. Since Naive Bayes and KNN gave 0, the final score became 0.
   - Majority Vote: No model voted for 182. It got 0 votes.
3. The "Intersection" Mistake (Code Bug)<br>
There is actually one method that did contain the correct answer, but your code marked it wrong anyway.
   - Look at Rank — Intersection. The result was a long list: {10, 103, ... 182, ...}.
   - Class 182 was inside that list!
   - However, your summary code checks for an exact match (==). Because the Intersection result is a list of many numbers, and 182 is just one number, the check if "182" == "{10, ... 182 ...}" returned False.



